# S1 · Arquitectura Big Data

**Integrante** · Leonardo Jesus Huaman Arhuata

**Dimensión** · Volumen de aeronaves en vuelo

Equipo 04 · AeroStreamPeru · Ciclo 9

## Mi aporte

En U1 uso PySpark sobre el Parquet de OpenSky extraído con Trino. En U2 el mismo dominio se observa con OpenSky REST, Kafka, Spark Structured Streaming y Grafana. Lambda permite mantener el cálculo histórico y actualizar el indicador con capturas nuevas.

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession

bronze = Path("/opt/s01-arquitectura/data/2026")
bloque = bronze / "01" / "2026-01-01_2026-01-05.parquet"

spark = (
    SparkSession.builder.master("local[2]")
    .appName("AeroStreamPeru-S01")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 15:07:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/24 15:07:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark 4.2.0


In [2]:
estados = spark.read.parquet(str(bloque))
estados.printSchema()
print(f"Filas: {estados.count():,} · Columnas: {len(estados.columns)}")
estados.select("time", "icao24", "onground", "lat", "lon").show(5)
estados.groupBy("onground").count().orderBy("onground").show()

root
 |-- time: long (nullable = true)
 |-- icao24: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- velocity: double (nullable = true)
 |-- heading: double (nullable = true)
 |-- vertrate: double (nullable = true)
 |-- callsign: string (nullable = true)
 |-- onground: boolean (nullable = true)
 |-- alert: boolean (nullable = true)
 |-- spi: boolean (nullable = true)
 |-- squawk: string (nullable = true)
 |-- baroaltitude: double (nullable = true)
 |-- geoaltitude: double (nullable = true)
 |-- lastposupdate: double (nullable = true)
 |-- lastcontact: double (nullable = true)
 |-- hour: long (nullable = true)



Filas: 2,365,631 · Columnas: 17


+----------+------+--------+------------------+------------------+
|      time|icao24|onground|               lat|               lon|
+----------+------+--------+------------------+------------------+
|1767466277|e8047a|   false|-0.912139892578125|-69.16382353184588|
|1767466278|e8047a|   false|-0.912139892578125|-69.16382353184588|
|1767466279|e8047a|   false|-0.912139892578125|-69.16382353184588|
|1767466280|e8047a|   false|-0.912139892578125|-69.16382353184588|
|1767466281|e8047a|   false|-0.912139892578125|-69.16382353184588|
+----------+------+--------+------------------+------------------+
only showing top 5 rows


[Stage 5:>                                                          (0 + 2) / 2]

+--------+-------+
|onground|  count|
+--------+-------+
|   false|2190502|
|    true| 175129|
+--------+-------+



## Interpretación

La tabla anterior cuenta filas de telemetría por estado de vuelo. Una aeronave aparece en muchas capturas. Mi indicador se construye contando identificadores distintos en cada captura y promediando los conteos durante la ventana.

In [3]:
spark.stop()